In [ ]:
from pathlib import Path
from typing import Callable
import lightning as L
import torch
import torch.nn.functional as F
from PIL import Image
from torch import nn
from torch.utils.data import DataLoader, Dataset, random_split
from torchvision import transforms
import os
import sys
from typing import Optional, Union
from torchmetrics import Accuracy
import mlflow
import subprocess
import zipfile
from lightning.pytorch.callbacks import EarlyStopping, ModelCheckpoint, BatchSizeFinder
from lightning.pytorch.tuner.tuning import Tuner

### Utils

In [ ]:
class CustomDataset(Dataset):
    encoding = {
        "dog": 0,
        "cat": 1,
    }
    decoding = list(encoding.values())

    def __init__(self, root_dir: Union[Path, str], transform: Optional[Callable] = None):
        self.root_dir = Path(root_dir)
        self.transform = transform
        self.paths = list(self.root_dir.iterdir())

    def __len__(self) -> int:
        return len(self.paths)

    def __getitem__(self, idx) -> tuple[torch.Tensor, int]:
        path = self.paths[idx]
        label = get_label(path)
        y = self.encoding[label]

        img = Image.open(path).convert('RGB')
        x = self.transform(img)
        return x, y
    
class CatVsDogsDataModule(L.LightningDataModule):
    def __init__(self, data_dir: Path, pct_train: float = 0.8, batch_size: int = 32):
        super().__init__()
        self.save_hyperparameters()
        self.data_dir = data_dir
        self.train_dir = data_dir / 'train'
        self.batch_size = batch_size
        self.pct_train = pct_train
        self.transform = transforms.Compose(
            [
                transforms.Resize((32, 32)),
                transforms.ToTensor(),
                transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
            ]
        ) 
    
    def prepare_data(self) -> None:
        self.data_dir.mkdir(exist_ok=True, parents=True)
        
        kaggle_path = Path.home() / ".kaggle" / "kaggle.json"
        if not kaggle_path.exists():
            raise Exception(f'{kaggle_path} does not exist, please add!')
        kaggle_path.chmod(0o600)

        subprocess.run(['kaggle', 'competitions', 'download', '-c', 'dogs-vs-cats', '-p', self.data_dir.as_posix()], check=True)
        with zipfile.ZipFile(self.data_dir / 'dogs-vs-cats.zip', 'r') as f:
            f.extractall(self.data_dir)
        with zipfile.ZipFile(self.data_dir / 'train.zip', 'r') as f:
            f.extractall(self.data_dir)
        assert len(list((self.train_dir).iterdir())) == 25000
        print("Dataset downloaded and extracted successfully.")
    
    def setup(self, stage: str) -> None:
        if stage == 'fit':
            ds = CustomDataset(root_dir=self.train_dir, transform=self.transform)
            self.train_ds, self.valid_ds = split_dataset(ds, pct_train=self.pct_train)
    
    def train_dataloader(self) -> DataLoader:
        return DataLoader(self.train_ds, batch_size=self.batch_size, shuffle=True)

    def val_dataloader(self) -> DataLoader:
        return DataLoader(self.valid_ds, batch_size=self.batch_size, shuffle=False)

class ConvNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.layer1 = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
        )
        self.layer2 = nn.Sequential(
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
        )
        self.fc1 = nn.Linear(8 * 8 * 64, 1000)
        self.fc2 = nn.Linear(1000, 2)

    def forward(self, x):
        out = self.layer1(x)
        out = self.layer2(out)
        out = out.reshape(out.size(0), -1)
        out = self.fc1(out)
        out = self.fc2(out)
        return out


class ImageClassifier(L.LightningModule):
    def __init__(self, model: nn.Module, lr: float = 1e-3):
        super().__init__()
        self.save_hyperparameters()
        self.model = model
        self.lr = lr
        self.accuracy = Accuracy(task='binary')

    def forward(self, x):
        out = self.model.forward(x)
        return out

    def _step(self, batch, batch_idx, set_name: str):
        x, y = batch
        yprob = self.model.forward(x)
        loss = F.cross_entropy(yprob, y)
        yhat = yprob.argmax(-1)
        acc = self.accuracy(yhat, y)
        self.log(f"{set_name}_loss", loss, on_step=True, on_epoch=True, prog_bar=True)
        self.log(f"{set_name}_acc", acc, on_step=True, on_epoch=True, prog_bar=True)
        return loss

    def training_step(self, batch, batch_idx):
        return self._step(batch, batch_idx, "train")

    def validation_step(self, batch, batch_idx):
        return self._step(batch, batch_idx, "valid")

    def configure_optimizers(self):
        return torch.optim.Adam(self.parameters(), lr=self.lr)


class ModelTuner:
    def __init__(self, trainer: L.Trainer, model: L.LightningModule, data_module: L.LightningDataModule):
        self.tuner = Tuner(trainer)
        self.model = model
        self.data_module = data_module

    def find_batch_size(self):
        self.tuner.scale_batch_size(self.model, datamodule=self.data_module)

    def find_learning_rate(self):
        lr_finder = self.tuner.lr_find(self.model, datamodule=self.data_module)
        fig = lr_finder.plot(suggest=True)
        self.lr = lr_finder.suggestion()
        fig.show()


def split_dataset(ds: Dataset, pct_train: float) -> tuple[Dataset, Dataset]:
    train_size = int(pct_train * len(ds))
    val_size = len(ds) - train_size
    train_ds, valid_ds = random_split(ds, [train_size, val_size])
    return train_ds, valid_ds

def get_label(path: Path) -> str:
    return path.stem.split(".")[0]

### Tests

In [ ]:
def test_get_label():
    path = Path('/workspaces/example_cdk/data/train/dog.8011.jpg')
    assert get_label(path) == 'dog'

def test_trainer_fast_dev_run():
    trainer = L.Trainer(fast_dev_run=True)
    trainer.fit(lit_conv, train_dl, valid_dl)

def test_trainer_overfit_batches():
    trainer = L.Trainer(overfit_batches=1, max_epochs=50)
    trainer.fit(lit_conv, train_dl)
    final_accuracy = trainer.callback_metrics['train_acc_epoch'].item()
    assert final_accuracy > 0.99


### Config

In [ ]:
batch_size: int = 2048
root_path: str = '.'
pct_train: float = 0.8
lr: float = 1e-4
experiment_name = 'cats_vs_dogs'

root_dir = Path(root_path).resolve().parent
data_dir = root_dir / 'data'
torchscript_path = 'cat_vs_dogs.pt'

### Script

#### setting up experiment tracking

In [ ]:
mlflow_dir = (root_dir/'mlruns').as_posix()
mlflow.set_tracking_uri(f'file:{mlflow_dir}')
mlflow.set_experiment(experiment_name=experiment_name)
mlflow.pytorch.autolog()

In [ ]:
early_stopping = EarlyStopping('valid_loss')
model_checkpoint = ModelCheckpoint(monitor='valid_loss', filename='dogs-vs-cats-{epoch:03d}-{valid_loss:.3f}')
callbacks = [
    early_stopping,
    model_checkpoint,
]
with mlflow.start_run() as run:
    data_module = CatVsDogsDataModule(data_dir=data_dir, batch_size=batch_size)
    classifier = ConvNet()
    model = ImageClassifier(model=classifier, lr=lr)
    trainer = L.Trainer(max_epochs=1, callbacks=callbacks)
    trainer.fit(model, datamodule=data_module)
    
    best_model = ImageClassifier.load_from_checkpoint(model_checkpoint.best_model_path)
    mlflow.pytorch.log_model(best_model, 'model')
    mlflow.log_params(data_module.hparams)
    mlflow.log_params(best_model.hparams)

    scripted_model = torch.jit.script(best_model.model)
    scripted_model.save(torchscript_path)
    mlflow.log_artifact(torchscript_path)